In [1]:
import os
import random
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


In [2]:
df = pd.read_csv("dataset_with_latlon_reindexed.csv")
df.columns = df.columns.str.strip()


In [3]:
def convert_price(price):
    try:
        raw = str(price).replace("?", "").replace(",", "").strip().lower()
        value = float(re.sub(r"[^0-9.]", "", raw))
        if "cr" in raw:
            return value * 1e7
        if re.search(r"\bl\b|lac|lakh", raw):
            return value * 1e5
        return value
    except Exception:
        return None

df["Price"] = df["Price"].apply(convert_price)
df = df.dropna(subset=["Price"])
df = df[df["Price"] > 0].copy()

df["log_price"] = np.log1p(df["Price"])


In [4]:
# Extract listing features available at prediction time.
df["BHK"] = df["Property Title"].str.extract(r"(\d+)\s*BHK", flags=re.I).astype(float)
df["Balcony"] = df["Balcony"].map({"Yes": 1, "No": 0}).fillna(0)

for col in ["BHK", "Baths", "Total_Area", "latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

df["BHK"] = df["BHK"].clip(lower=1, upper=12)
df["Baths"] = df["Baths"].clip(lower=1, upper=12)
df["Total_Area"] = df["Total_Area"].clip(lower=100)


In [5]:
# Do not use Price-derived features here; they leak the target into training.
df["bhk_density"] = df["BHK"] / (df["Total_Area"] + 1)
df["bath_per_bhk"] = df["Baths"] / (df["BHK"] + 1)
df["area_per_bhk"] = df["Total_Area"] / (df["BHK"] + 1)

coords = df[["latitude", "longitude"]]
kmeans = KMeans(n_clusters=10, random_state=SEED, n_init=10)
df["location_cluster"] = kmeans.fit_predict(coords)


In [6]:
# Fit TF-IDF only on train data after the split to avoid validation leakage.
tfidf = TfidfVectorizer(max_features=150, min_df=2, ngram_range=(1, 2))


In [7]:
features = [
    "BHK",
    "Baths",
    "Balcony",
    "Total_Area",
    "latitude",
    "longitude",
    "bhk_density",
    "location_cluster",
    "bath_per_bhk",
    "area_per_bhk",
]

y = df["log_price"].values


In [8]:
scaler = StandardScaler()


In [9]:
image_dir = os.path.join(os.getcwd(), "images_reindexed")

df["image_path"] = df.index.map(
    lambda x: os.path.join(image_dir, f"{x}.png")
)


In [10]:
def make_price_bins(frame, q=8):
    try:
        return pd.qcut(frame["log_price"], q=q, labels=False, duplicates="drop")
    except ValueError:
        return None

# Stratify by price band so train/val/test see similar cheap/mid/expensive distributions.
price_bins = make_price_bins(df, q=8)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=price_bins,
)

temp_bins = make_price_bins(temp_df, q=4)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_bins,
)

scaler.fit(train_df[features])
train_tab = scaler.transform(train_df[features])
val_tab = scaler.transform(val_df[features])
test_tab = scaler.transform(test_df[features])

tfidf.fit(train_df["Description"].fillna(""))
train_text = tfidf.transform(train_df["Description"].fillna("")).toarray()
val_text = tfidf.transform(val_df["Description"].fillna("")).toarray()
test_text = tfidf.transform(test_df["Description"].fillna("")).toarray()

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Price range train:", int(train_df["Price"].min()), "to", int(train_df["Price"].max()))
print("Price range val:  ", int(val_df["Price"].min()), "to", int(val_df["Price"].max()))
print("Price range test: ", int(test_df["Price"].min()), "to", int(test_df["Price"].max()))


Train: 10166 | Val: 2178 | Test: 2179
Price range train: 1 to 840000000
Price range val:   150000 to 402000000
Price range test:  100000 to 350000000


In [11]:
class MultiModalDataset(Dataset):
    def __init__(self, df, tab, text, transform):
        self.df = df.reset_index(drop=True)
        self.tab = tab
        self.text = text
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"]).convert("RGB")
        img = self.transform(img)

        tab = torch.tensor(self.tab[idx], dtype=torch.float32)
        txt = torch.tensor(self.text[idx], dtype=torch.float32)
        y = torch.tensor(row["log_price"], dtype=torch.float32)

        return img, tab, txt, y


In [12]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08)
    ], p=0.35),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [13]:
batch_size = 32
num_workers = 0
pin_memory = torch.cuda.is_available()

train_ds = MultiModalDataset(train_df, train_tab, train_text, train_transform)
val_ds = MultiModalDataset(val_df, val_tab, val_text, eval_transform)
test_ds = MultiModalDataset(test_df, test_tab, test_text, eval_transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)


In [14]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, hidden_size, num_heads=8, dropout=0.25):
        super().__init__()
        self.query_norm = nn.LayerNorm(hidden_size)
        self.context_norm = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.ff_norm = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size),
        )

    def forward(self, query_token, context_tokens):
        attn_out, _ = self.attn(
            self.query_norm(query_token),
            self.context_norm(context_tokens),
            self.context_norm(context_tokens),
            need_weights=False,
        )
        token = query_token + self.dropout(attn_out)
        token = token + self.dropout(self.ff(self.ff_norm(token)))
        return token


class FusionModel(nn.Module):
    def __init__(self, tab_size, text_size, hidden_size=256, dropout=0.35, num_heads=8):
        super().__init__()

        self.cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        cnn_features = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        for p in self.cnn.parameters():
            p.requires_grad = False

        # Cross-attention needs stronger image features, but keep fine-tuning controlled.
        for name, param in self.cnn.named_parameters():
            if name.startswith("layer3") or name.startswith("layer4"):
                param.requires_grad = True

        self.image_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_features, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )

        self.tab_head = nn.Sequential(
            nn.Linear(tab_size, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )

        self.text_head = nn.Sequential(
            nn.Linear(text_size, 192),
            nn.BatchNorm1d(192),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(192, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 3,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.self_attention = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # Explicit cross-attention: each modality asks the other two modalities for useful context.
        self.image_cross = CrossAttentionBlock(hidden_size, num_heads=num_heads, dropout=dropout)
        self.tab_cross = CrossAttentionBlock(hidden_size, num_heads=num_heads, dropout=dropout)
        self.text_cross = CrossAttentionBlock(hidden_size, num_heads=num_heads, dropout=dropout)

        self.gate = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 3),
            nn.Softmax(dim=1),
        )

        self.regressor = nn.Sequential(
            nn.Linear(hidden_size * 4, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, img, tab, text):
        img_token = self.image_head(self.cnn(img)).unsqueeze(1)
        tab_token = self.tab_head(tab).unsqueeze(1)
        text_token = self.text_head(text).unsqueeze(1)

        tokens = torch.cat([img_token, tab_token, text_token], dim=1)
        tokens = self.self_attention(tokens)

        img_token = self.image_cross(tokens[:, 0:1, :], tokens[:, 1:3, :])
        tab_token = self.tab_cross(tokens[:, 1:2, :], torch.cat([tokens[:, 0:1, :], tokens[:, 2:3, :]], dim=1))
        text_token = self.text_cross(tokens[:, 2:3, :], tokens[:, 0:2, :])

        fused_tokens = torch.cat([img_token, tab_token, text_token], dim=1)
        flat_tokens = fused_tokens.flatten(start_dim=1)
        gate_weights = self.gate(flat_tokens).unsqueeze(-1)
        weighted_summary = (fused_tokens * gate_weights).sum(dim=1)

        final_features = torch.cat([flat_tokens, weighted_summary], dim=1)
        return self.regressor(final_features)


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = FusionModel(
    tab_size=len(features),
    text_size=train_text.shape[1],
    hidden_size=256,
    dropout=0.35,
    num_heads=8,
).to(device)

criterion = nn.SmoothL1Loss(beta=0.30)

backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("cnn."):
        backbone_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params, "lr": 7e-5},
    ],
    weight_decay=2e-4,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

artifact_dir = Path("model_artifacts_cross_attention")
artifact_dir.mkdir(exist_ok=True)
best_model_path = artifact_dir / "fusion2_cross_attention_best_model.pth"
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


Using device: cuda
Trainable parameters: 14710212


In [17]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    preds, actuals = [], []
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for imgs, tabs, texts, targets in loader:
            imgs = imgs.to(device, non_blocking=True)
            tabs = tabs.to(device, non_blocking=True)
            texts = texts.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True).unsqueeze(1)

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(imgs, tabs, texts)
            loss = criterion(outputs, targets)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            preds.extend(outputs.detach().cpu().numpy().flatten())
            actuals.extend(targets.detach().cpu().numpy().flatten())

    avg_loss = total_loss / len(loader.dataset)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    r2 = r2_score(actuals, preds)
    return avg_loss, rmse, r2

max_epochs = 60
patience = 8
best_val_loss = float("inf")
bad_epochs = 0
history = []

for epoch in range(1, max_epochs + 1):
    train_loss, train_rmse, train_r2 = run_epoch(train_loader, training=True)
    val_loss, val_rmse, val_r2 = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)
    current_lrs = [group["lr"] for group in optimizer.param_groups]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_rmse": train_rmse,
        "val_rmse": val_rmse,
        "train_r2": train_r2,
        "val_r2": val_r2,
        "backbone_lr": current_lrs[0],
        "head_lr": current_lrs[1],
    })

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
        f"train_rmse={train_rmse:.4f} val_rmse={val_rmse:.4f} | "
        f"train_r2={train_r2:.4f} val_r2={val_r2:.4f} | "
        f"lr={current_lrs[0]:.1e}/{current_lrs[1]:.1e}"
    )

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        bad_epochs = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "features": features,
            "text_size": train_text.shape[1],
            "tab_size": len(features),
            "hidden_size": 256,
            "dropout": 0.35,
            "num_heads": 8,
            "best_val_loss": best_val_loss,
            "epoch": epoch,
            "architecture": "cross_attention_fusion_v1",
        }, best_model_path)
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch}. Best val_loss={best_val_loss:.4f}")
            break

joblib.dump(scaler, artifact_dir / "scaler.joblib")
joblib.dump(tfidf, artifact_dir / "tfidf.joblib")
joblib.dump(kmeans, artifact_dir / "kmeans.joblib")
pd.DataFrame(history).to_csv(artifact_dir / "fusion2_cross_attention_training_history.csv", index=False)

checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Loaded best cross-attention checkpoint from epoch", checkpoint["epoch"])


Epoch 01 | train_loss=11.5036 val_loss=6.7985 | train_rmse=11.9389 val_rmse=7.0048 | train_r2=-163.6116 val_r2=-62.2376 | lr=1.0e-05/7.0e-05
Epoch 02 | train_loss=2.8607 val_loss=0.4243 | train_rmse=3.8397 val_rmse=0.7224 | train_r2=-16.0261 val_r2=0.3275 | lr=1.0e-05/7.0e-05
Epoch 03 | train_loss=0.6665 val_loss=0.3764 | train_rmse=1.0611 val_rmse=0.6576 | train_r2=-0.3004 val_r2=0.4426 | lr=1.0e-05/7.0e-05
Epoch 04 | train_loss=0.6338 val_loss=0.3407 | train_rmse=1.0144 val_rmse=0.6050 | train_r2=-0.1884 val_r2=0.5283 | lr=1.0e-05/7.0e-05
Epoch 05 | train_loss=0.5931 val_loss=0.2685 | train_rmse=0.9684 val_rmse=0.5374 | train_r2=-0.0830 val_r2=0.6278 | lr=1.0e-05/7.0e-05
Epoch 06 | train_loss=0.5695 val_loss=0.3544 | train_rmse=0.9451 val_rmse=0.6152 | train_r2=-0.0316 val_r2=0.5122 | lr=1.0e-05/7.0e-05
Epoch 07 | train_loss=0.5392 val_loss=0.2466 | train_rmse=0.9029 val_rmse=0.5059 | train_r2=0.0586 val_r2=0.6701 | lr=1.0e-05/7.0e-05
Epoch 08 | train_loss=0.5156 val_loss=0.2526 | tr

In [18]:
def collect_predictions(loader):
    model.eval()
    preds, actuals = [], []

    with torch.no_grad():
        for imgs, tabs, texts, targets in loader:
            imgs = imgs.to(device, non_blocking=True)
            tabs = tabs.to(device, non_blocking=True)
            texts = texts.to(device, non_blocking=True)
            outputs = model(imgs, tabs, texts)
            preds.extend(outputs.cpu().numpy().flatten())
            actuals.extend(targets.numpy().flatten())

    return np.array(preds), np.array(actuals)


def report_metrics(name, preds_log, actuals_log):
    rmse_log = np.sqrt(mean_squared_error(actuals_log, preds_log))
    mae_log = mean_absolute_error(actuals_log, preds_log)
    r2 = r2_score(actuals_log, preds_log)
    preds_price = np.expm1(preds_log)
    actuals_price = np.expm1(actuals_log)
    mae_price = mean_absolute_error(actuals_price, preds_price)
    rmse_price = np.sqrt(mean_squared_error(actuals_price, preds_price))
    mape = np.mean(np.abs((actuals_price - preds_price) / np.maximum(actuals_price, 1))) * 100

    print(f"{name} log RMSE: {rmse_log:.4f}")
    print(f"{name} log MAE:  {mae_log:.4f}")
    print(f"{name} R2:       {r2:.4f}")
    print(f"{name} price MAE:  Rs. {mae_price:,.0f}")
    print(f"{name} price RMSE: Rs. {rmse_price:,.0f}")
    print(f"{name} MAPE:     {mape:.2f}%")
    return {
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "r2": r2,
        "mae_price": mae_price,
        "rmse_price": rmse_price,
        "mape": mape,
    }


In [19]:
val_preds, val_actuals = collect_predictions(val_loader)
test_preds, test_actuals = collect_predictions(test_loader)

val_metrics = report_metrics("Validation", val_preds, val_actuals)
print("-" * 40)
test_metrics = report_metrics("Test", test_preds, test_actuals)

val_abs_pct_error = np.abs((np.expm1(val_actuals) - np.expm1(val_preds)) / np.maximum(np.expm1(val_actuals), 1)) * 100
confidence_calibration = {
    "p50_abs_pct_error": float(np.percentile(val_abs_pct_error, 50)),
    "p75_abs_pct_error": float(np.percentile(val_abs_pct_error, 75)),
    "p90_abs_pct_error": float(np.percentile(val_abs_pct_error, 90)),
}
joblib.dump(confidence_calibration, artifact_dir / "confidence_calibration.joblib")
joblib.dump({"validation": val_metrics, "test": test_metrics}, artifact_dir / "final_metrics.joblib")

print("-" * 40)
print("Confidence calibration from validation absolute percentage error:")
for key, value in confidence_calibration.items():
    print(f"{key}: {value:.2f}%")

print("Artifacts saved in:", artifact_dir.resolve())


Validation log RMSE: 0.4467
Validation log MAE:  0.3265
Validation R2:       0.7429
Validation price MAE:  Rs. 4,298,046
Validation price RMSE: Rs. 15,917,992
Validation MAPE:     39.36%
----------------------------------------
Test log RMSE: 0.4686
Test log MAE:  0.3327
Test R2:       0.7272
Test price MAE:  Rs. 4,665,436
Test price RMSE: Rs. 16,364,718
Test MAPE:     41.26%
----------------------------------------
Confidence calibration from validation absolute percentage error:
p50_abs_pct_error: 26.01%
p75_abs_pct_error: 47.02%
p90_abs_pct_error: 75.12%
Artifacts saved in: C:\Users\mrvin\anaconda3\Capstone\notebooks\model_artifacts_cross_attention
